# Eshmun Continued Pretraining — LoRA

Continues causal LM pretraining on `khairi/eshmun-pretraining` from `Khairi/Eshmun-1.3B-Base`
using **LoRA** adapters injected into the attention and FFN layers.

**Trainable parameters:**
- LoRA adapters on `q_proj`, `k_proj`, `v_proj`, `out_proj`, `fc1`, `fc2`
- `lm_head` — fully trainable (new tokens in extended vocab)
- `embed_tokens` — fully trainable (new tokens in extended vocab)
- All other base model weights: frozen

**Pipeline:**
1. Install dependencies
2. Login to HuggingFace Hub
3. Load model and tokenizer
4. Wrap with LoRA, unfreeze `lm_head` and `embed_tokens`
5. Load dataset (raw — tokenization deferred to collator)
6. Define `ProteinCLMCollator`
7. Train and push adapter weights to Hub

## 1. Install dependencies

In [ ]:
!pip install -q git+https://github.com/abidikhairi/eshmun.git
!pip install -q datasets transformers accelerate peft

## 2. Login to HuggingFace Hub

In [ ]:
from huggingface_hub import login as hf_login

hf_login()  # paste your HF write-access token when prompted

## 3. Imports

In [ ]:
from dataclasses import dataclass
from typing import Any

import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Configuration

In [ ]:
MODEL_ID   = "Khairi/Eshmun-1.3B-Base"
DATASET_ID = "khairi/eshmun-pretraining"
HF_REPO_ID = "khairi/Eshmun-1.3B-CPT-LoRA"   # adapter weights pushed here
OUTPUT_DIR = "/tmp/eshmun-1.3b-cpt-lora"

MAX_SEQ_LEN = 512

# LoRA hyperparameters
LORA_R           = 32
LORA_ALPHA       = 64
LORA_DROPOUT     = 0.05

# Training hyperparameters
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8       # effective batch = 32
NUM_TRAIN_EPOCHS            = 1
LEARNING_RATE               = 2e-4
WARMUP_STEPS                = 100
LOGGING_STEPS               = 50
SAVE_STEPS                  = 500

## 5. Model & tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float32,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_ID}  ({n_params / 1e6:.1f}M parameters)")

## 6. LoRA setup

LoRA adapters are injected into every attention projection and FFN linear layer.
After wrapping, `lm_head` and `embed_tokens` are explicitly unfrozen so the
extended-vocabulary embeddings are updated during training.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
)

model = get_peft_model(model, lora_config)

# Unfreeze lm_head and embed_tokens — new tokens must be learned from scratch
for name, param in model.named_parameters():
    if "lm_head" in name or "embed_tokens" in name:
        param.requires_grad = True

model.print_trainable_parameters()

## 7. Dataset

Raw dataset — no upfront tokenization. The `ProteinCLMCollator` handles everything at batch time.

In [ ]:
dataset = load_dataset(DATASET_ID, split="train")
dataset = dataset.select_columns(["Content"])

print(dataset)

## 8. Data collator

In [ ]:
@dataclass
class ProteinCLMCollator:
    tokenizer: PreTrainedTokenizerBase
    max_length: int = 512

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        texts = [f["Content"] for f in features]
        encoded = self.tokenizer(
            texts,
            max_length=self.max_length,
            truncation=True,
            padding="longest",
            return_tensors="pt",
        )
        labels = encoded["input_ids"].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        encoded["labels"] = labels
        return encoded


collator = ProteinCLMCollator(tokenizer=tokenizer, max_length=MAX_SEQ_LEN)

## 9. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    fp16=False,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    push_to_hub=True,
    hub_model_id=HF_REPO_ID,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=collator,
)

trainer.train()

## 10. Save and push to Hub

`model.save_pretrained` saves only the **adapter weights** (not the full base model),
keeping the checkpoint small. Load it back with:
```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained("Khairi/Eshmun-1.3B-Base", trust_remote_code=True)
model = PeftModel.from_pretrained(base, "khairi/Eshmun-1.3B-CPT-LoRA")
```

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

trainer.push_to_hub(commit_message="continued pretraining lora checkpoint")
print(f"Adapter pushed to https://huggingface.co/{HF_REPO_ID}")